# ETF Portfolio Exploration

This notebook demonstrates a practical portfolio-research workflow using the local TipRanks ETF database.

Example portfolio:

- ASHR
- FXI
- MCHI
- EWS
- ARTY
- BOTZ
- SOXX
- INDA
- SPY

## Goals

1. Load a sample multi-region and thematic ETF portfolio.
2. Check which ETFs are already available in the local database.
3. Compare holdings overlap for the available subset.
4. Estimate stock-level exposure after ETF weights are combined.
5. Highlight the missing ETFs that still need data ingestion.

In [ ]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / "tipranks_api").exists() and (root / "data").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tipranks_api import TipRanksQueryDB

DB_PATH = PROJECT_ROOT / "data" / "etf_database.db"
query = TipRanksQueryDB(str(DB_PATH))

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

DB_PATH

In [ ]:
example_portfolio = {
    "ASHR": 0.10,
    "FXI": 0.10,
    "MCHI": 0.10,
    "EWS": 0.05,
    "ARTY": 0.10,
    "BOTZ": 0.10,
    "SOXX": 0.15,
    "INDA": 0.10,
    "SPY": 0.20,
}

portfolio_df = pd.DataFrame(
    {"ETF": list(example_portfolio.keys()), "Target Weight": list(example_portfolio.values())}
)
portfolio_df["Target Weight"] = portfolio_df["Target Weight"].map(lambda value: f"{value:.0%}")
portfolio_df

In [ ]:
coverage = query.summarize_requested_etfs(list(example_portfolio.keys()))
coverage

In [ ]:
available_etfs = coverage.loc[coverage["available"], "requested_ticker"].tolist()
missing_etfs = coverage.loc[~coverage["available"], "requested_ticker"].tolist()

available_allocations = {
    ticker: weight
    for ticker, weight in example_portfolio.items()
    if ticker in available_etfs
}

renormalized_allocations = {}
if available_allocations:
    total_available_weight = sum(available_allocations.values())
    renormalized_allocations = {
        ticker: weight / total_available_weight
        for ticker, weight in available_allocations.items()
    }

pd.DataFrame(
    {
        "available_etfs": pd.Series(available_etfs),
        "missing_etfs": pd.Series(missing_etfs),
    }
)

## Holdings Overlap For The Available Subset

If some requested ETFs are missing from the database, the analysis below still works on the subset that is currently available.

In [ ]:
if available_etfs:
    top_overlap = query.get_top_holdings(available_etfs, n=15)
    top_overlap.head(25)
else:
    pd.DataFrame({"message": ["No requested ETFs are currently available in the local database."]})

## Stock-Level Portfolio Look-Through

The next view translates ETF weights into estimated stock-level exposures using the available ETF subset. We renormalize the available weights to keep the analysis internally consistent.

In [ ]:
if renormalized_allocations:
    lookthrough = query.get_portfolio_weight(renormalized_allocations, top=25)
    lookthrough
else:
    pd.DataFrame({"message": ["No available ETF weights to analyze."]})

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    etf_universe = pd.read_sql_query(
        "SELECT etf_ticker, etf_name, holdings, smart_score, one_year_return FROM etf_info ORDER BY etf_ticker",
        conn,
    )

etf_universe.head(20)

## Optional: Fetch Missing ETFs

If you have valid TipRanks credentials stored in environment variables, you can extend the local database with the missing ETFs.

Suggested environment variables:

- `TIPRANKS_EMAIL`
- `TIPRANKS_PASSWORD`

In [ ]:
# import os
# from tipranks_api import TipRanksAPI, TipRanksDB
#
# email = os.getenv("TIPRANKS_EMAIL")
# password = os.getenv("TIPRANKS_PASSWORD")
#
# if email and password and missing_etfs:
#     api = TipRanksAPI(email, password)
#     db = TipRanksDB(str(DB_PATH))
#     new_etfs = api.fetch_etf_data(missing_etfs)
#     db.store_etf_data(new_etfs)
#
#     for etf in new_etfs:
#         holdings = api.fetch_etf_holdings(etf["etf_ticker"], etf["holdings"])
#         db.store_etf_holdings(holdings)
#
#     db.extract_unique_holdings()
#     db.close()
#     print("Database updated with missing ETFs.")
# else:
#     print("Set credentials and ensure there are missing ETFs before running this cell.")